## Report
```
The program generates a random DNA sequence of 24,000,000 nucleotides and maps each position to its codon ID, warp ID, and lane ID inside the kernel. The first 96 threads write their mapping records to an array. Every lane-0 thread atomically increments a global warp counter, and every codon-start thread (index divisible by 3) atomically increments one of 64 codon frequency bins, covering all AAA–TTT triplets. The kernel is tested across block sizes 64, 128, 256, and 512.
```

In [7]:
%%writefile codon_warp_mapping_analysis.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <time.h>

#define N 24000000
#define WARP_SIZE 32
#define RECORD_COUNT 96

struct MappingRecord {
    int gid;
    int codon_id;
    int warp_id;
    int lane_id;
};

__global__ void codon_warp_mapping_analysis_kernel(const int *seq, MappingRecord *records, unsigned long long *warp_counter, unsigned long long *codon_counts, int n, int record_count)
{
    int gid = blockIdx.x * blockDim.x + threadIdx.x;

    if (gid >= n) {
        return;
    }

    int codon_id = gid / 3;
    int warp_id = threadIdx.x / WARP_SIZE;
    int lane_id = threadIdx.x % WARP_SIZE;

    if (gid < record_count) {
        records[gid].gid = gid;
        records[gid].codon_id = codon_id;
        records[gid].warp_id = warp_id;
        records[gid].lane_id = lane_id;
    }

    if (lane_id == 0) {
        atomicAdd(warp_counter, 1ULL);
    }

    if (gid % 3 == 0 && gid + 2 < n) {
        int codon_idx = seq[gid] * 16 + seq[gid + 1] * 4 + seq[gid + 2];

        atomicAdd(&codon_counts[codon_idx], 1ULL);
    }
}

static void print_codon_counts(const unsigned long long *counts)
{
    const char *bases = "ACGT";

    printf("%-6s  %s\n", "Codon", "Count");
    printf("%-6s  %s\n", "------", "----------");

    for (int i = 0; i < 64; i++) {
        char codon[4] = {bases[(i >> 4) & 3], bases[(i >> 2) & 3], bases[i & 3], '\0'};

        printf("%-6s  %llu\n", codon, counts[i]);
    }
}

void launch_dna_kernel(const int *h_seq, int block_size)
{
    printf("\nBlock size : %-4d\n", block_size);

    int grid_size = (N + block_size - 1) / block_size;
    long long total_threads = (long long)grid_size * block_size;
    long long total_codons = N / 3;

    int *d_seq = NULL;
    MappingRecord *d_records = NULL;
    unsigned long long *d_warp_counter = NULL;
    unsigned long long *d_codon_counts = NULL;

    cudaMalloc((void **)&d_seq, N * sizeof(int));
    cudaMalloc((void **)&d_records, RECORD_COUNT * sizeof(MappingRecord));
    cudaMalloc((void **)&d_warp_counter, sizeof(unsigned long long));
    cudaMalloc((void **)&d_codon_counts, 64 * sizeof(unsigned long long));

    cudaMemset(d_warp_counter, 0, sizeof(unsigned long long));
    cudaMemset(d_codon_counts, 0, 64 * sizeof(unsigned long long));

    cudaMemcpy(d_seq, h_seq, N * sizeof(int), cudaMemcpyHostToDevice);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    codon_warp_mapping_analysis_kernel<<<grid_size, block_size>>>(d_seq, d_records, d_warp_counter, d_codon_counts, N, RECORD_COUNT);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms = 0.0f;
    cudaEventElapsedTime(&ms, start, stop);

    MappingRecord h_records[RECORD_COUNT];
    unsigned long long h_warp_counter = 0;
    unsigned long long h_codon_counts[64] = {0};

    cudaMemcpy(h_records, d_records, RECORD_COUNT * sizeof(MappingRecord), cudaMemcpyDeviceToHost);
    cudaMemcpy(&h_warp_counter, d_warp_counter, sizeof(unsigned long long), cudaMemcpyDeviceToHost);
    cudaMemcpy(h_codon_counts, d_codon_counts, 64 * sizeof(unsigned long long), cudaMemcpyDeviceToHost);

    printf("Total codons : %lld\n", total_codons);
    printf("Total threads launched : %lld\n", total_threads);
    printf("Total warps: %llu\n", h_warp_counter);
    printf("Number of blocks : %d\n", grid_size);
    printf("Kernel execution time : %.4f ms\n", ms);
    print_codon_counts(h_codon_counts);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_seq);
    cudaFree(d_records);
    cudaFree(d_warp_counter);
    cudaFree(d_codon_counts);
}

int main(void)
{
    srand((unsigned int)time(NULL));

    int *h_seq = (int *)malloc(N * sizeof(int));

    for (int i = 0; i < N; i++) {
        h_seq[i] = rand() % 4;
    }

    int block_sizes[] = {64, 128, 256, 512};
    int num_configs = sizeof(block_sizes) / sizeof(block_sizes[0]);

    for (int i = 0; i < num_configs; i++) {
        launch_dna_kernel(h_seq, block_sizes[i]);
    }

    free(h_seq);

    return 0;
}

Overwriting codon_warp_mapping_analysis.cu


In [8]:
!nvcc -arch=sm_75 codon_warp_mapping_analysis.cu -o codon_warp_mapping_analysis

In [9]:
!./codon_warp_mapping_analysis


Block size : 64  
Total codons : 8000000
Total threads launched : 24000000
Total warps: 750000
Number of blocks : 375000
Kernel execution time : 5.0417 ms
Codon   Count
------  ----------
AAA     125464
AAC     125006
AAG     124850
AAT     124933
ACA     125037
ACC     124614
ACG     125799
ACT     124881
AGA     125485
AGC     125271
AGG     124742
AGT     124744
ATA     124567
ATC     125450
ATG     124607
ATT     124332
CAA     124949
CAC     125663
CAG     125725
CAT     124112
CCA     125240
CCC     124758
CCG     125016
CCT     124953
CGA     124564
CGC     124270
CGG     125601
CGT     125230
CTA     124518
CTC     125188
CTG     124393
CTT     125738
GAA     125309
GAC     124860
GAG     125228
GAT     124852
GCA     125672
GCC     124393
GCG     124729
GCT     124783
GGA     124576
GGC     124944
GGG     125180
GGT     125057
GTA     124669
GTC     125018
GTG     124721
GTT     124984
TAA     124886
TAC     124835
TAG     125227
TAT     124828
TCA     125201
TCC     125056
T